In [15]:
import itertools
import concurrent.futures
import constants as Cs
import os
import json
import datetime
import pickle
import optuna
import numpy as np
from scipy.spatial.distance import pdist
import logging


SEEDS = [101,102,103]
TEST_EVAL_EPS = 5
# lambda fit archving [FrozenTrial(number=80, state=<TrialState.COMPLETE: 1>, values=[118.11550385203829], datetime_start=datetime.datetime(2026, 5, 22, 5, 4, 53, 822616), datetime_complete=datetime.datetime(2026, 5, 22, 5, 36, 8, 359073), params={'crossmethod': 'uniform', 'lambda': 70, 'mu': 70, 'mutation_rate': 0.01, 'cross_rate': 0.9000000000000001, 'sigma': 0.5, 'archiving_period': 4, 'archive_batch': 5, 'cross': 0.6}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'crossmethod': CategoricalDistribution(choices=('uniform', 'mean')), 'lambda': CategoricalDistribution(choices=(40, 50, 60, 70)), 'mu': CategoricalDistribution(choices=(40, 50, 60, 70)), 'mutation_rate': FloatDistribution(high=0.1, log=False, low=0.0, step=0.01), 'cross_rate': FloatDistribution(high=1.0, log=False, low=0.3, step=0.1), 'sigma': FloatDistribution(high=3.0, log=False, low=0.5, step=0.5), 'archiving_period': IntDistribution(high=5, log=False, low=2, step=1), 'archive_batch': IntDistribution(high=5, log=False, low=1, step=1), 'cross': FloatDistribution(high=0.9, log=False, low=0.1, step=0.1)}, trial_id=81, value=None), FrozenTrial(number=86, state=<TrialState.COMPLETE: 1>, values=[118.11550385203829], datetime_start=datetime.datetime(2026, 5, 22, 5, 36, 31, 908497), datetime_complete=datetime.datetime(2026, 5, 22, 6, 11, 5, 128793), params={'crossmethod': 'uniform', 'lambda': 70, 'mu': 70, 'mutation_rate': 0.01, 'cross_rate': 0.9000000000000001, 'sigma': 0.5, 'archiving_period': 4, 'archive_batch': 5, 'cross': 0.6}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'crossmethod': CategoricalDistribution(choices=('uniform', 'mean')), 'lambda': CategoricalDistribution(choices=(40, 50, 60, 70)), 'mu': CategoricalDistribution(choices=(40, 50, 60, 70)), 'mutation_rate': FloatDistribution(high=0.1, log=False, low=0.0, step=0.01), 'cross_rate': FloatDistribution(high=1.0, log=False, low=0.3, step=0.1), 'sigma': FloatDistribution(high=3.0, log=False, low=0.5, step=0.5), 'archiving_period': IntDistribution(high=5, log=False, low=2, step=1), 'archive_batch': IntDistribution(high=5, log=False, low=1, step=1), 'cross': FloatDistribution(high=0.9, log=False, low=0.1, step=0.1)}, trial_id=87, value=None)]
# lambda - this one is bad actually novelty [FrozenTrial(number=4, state=<TrialState.COMPLETE: 1>, values=[-9.291787437438707], datetime_start=datetime.datetime(2026, 5, 19, 22, 25, 13, 886515), datetime_complete=datetime.datetime(2026, 5, 19, 23, 0, 16, 677521), params={'crossmethod': 'mean', 'lambda': 60, 'mu': 60, 'mutation_rate': 0.18, 'cross_rate': 1.0, 'sigma': 2.5}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'crossmethod': CategoricalDistribution(choices=('uniform', 'mean')), 'lambda': CategoricalDistribution(choices=(40, 50, 60, 70)), 'mu': CategoricalDistribution(choices=(40, 50, 60, 70)), 'mutation_rate': FloatDistribution(high=0.5, log=False, low=0.0, step=0.01), 'cross_rate': FloatDistribution(high=1.0, log=False, low=0.3, step=0.1), 'sigma': FloatDistribution(high=3.0, log=False, low=0.5, step=0.5)}, trial_id=208, value=None)]
EXAMPLE_MAPPING  = {
    "lambda":"l",
    "mu": "m",
    "mutation_rate":"mr",
    "cross_rate":"cr",
    "sigma": "mutation_sigma",
    "cross":"cross_uni",
    "crossmethod":"cross_method"
}

def rename(dict):
    return {EXAMPLE_MAPPING.get(k, k): v for k, v in dict.items()}

def task_job(en, alg, container, args, s, out_path):
    env = Cs.ENIVROMENTS[en]()
    df, pop = Cs.ALG_MAPPING[alg].argumented_function(env=en, container=container, seed=s, out_path=out_path, **args)
    print("Testing " + str(s))
    fitnesses = [env.evalutation_b(p, 42, TEST_EVAL_EPS) for p in pop]
    print("Finished seed %d of algorithm %s" % (s, alg))
    return fitnesses, pop


def evaluation_of_setup(en, run_name, alg, container, out_path, ev_ng, **kwargs):
    #we do not deviate the sigma
    # first we evaluate the current setup

    stat_futures = {}
    args = rename(kwargs)
    args["ng"] = ev_ng
    dirpath = os.path.join(os.path.realpath(out_path), container,alg, run_name)

    with concurrent.futures.ProcessPoolExecutor(max_workers=5) as executor:
        print("Launching " + alg + "on Enviroment " + str(en) + f" with {args}")
        for s in SEEDS:
            future = executor.submit(task_job, container=container, alg=alg, en=en, args=args, s=s, out_path=dirpath + "/stat")
            stat_futures[future] = s

    stats = {}
    pops = {}
    maxes = []
    diversities = []
    for future in concurrent.futures.as_completed(stat_futures):
        s = stat_futures[future]
        result = future.result()
        if "fitness" not in stats:
            stats["fitness"] = {}
        stats["fitness"][s] = result[0]
        behaviors = list(map(lambda x:x[1], result[0]))
        fitnesses = list(map(lambda x:x[0], result[0]))
        maxes.append(np.max(fitnesses))
        diversity = pdist(np.array(behaviors)).mean()
        diversities.append(diversity)
        pops[s]= result[1] 
    ts = datetime.datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    filename = f"{ts}_{container}_"+ "|".join(f"{k}: {v}" for k, v in args.items())
    stats["environment"] = en
    stats["algorithm"] = alg
    stats["container"] = container
    stats["arguments"] = args
    json_path = os.path.join(dirpath, filename +".json")

    with open(json_path, "w") as json_file:
        json.dump(stats,json_file)
        print("Finished "+ filename)
    return np.mean(maxes), np.mean(diversity)
  



In [16]:

from concurrent.futures import Future
def select_minimal_exaples(examples):
    pop = np.inf
    selected_trials = []
    selected_trials_index = []
    for i, t in enumerate(examples):
        if "lambda" in t:
            k = t["lambda"]
            if "mu" in t:
                k+=t["mu"]
        elif "pop" in t:
            k=t["pop"]
        else: raise NameError(f"wtf")
        if pop == k:
            selected_trials.append(t)
        elif pop > k:
            pop = k
            selected_trials = [t]
            selected_trials_index.append(i)

    return selected_trials, selected_trials_index

def argument_combination_genration(selected, D_args):
    deltas = []
    contraints_global = []
    for i, a in enumerate(D_args):
        delta = D_args[a][0]
        constraints = D_args[a][1]
        deltas.append(delta)
        contraints_global.append(constraints)

    generated_arguments = []

    for delta in itertools.product(*deltas):
        new = selected.copy()
        keep = True
        for i, a in enumerate(D_args):
            new[a] = selected[a] + delta[i]
            passed = (len(contraints_global[i]) == 0 or
            (contraints_global[i][0] is None or
            contraints_global[i][0] < new[a]) and
            (contraints_global[i][1] is None or
            contraints_global[i][1] > new[a]))
            keep = keep and passed

        if not keep: 
            print(f"Leaving out {new}")
            continue
    

        generated_arguments.append(new)
    return generated_arguments

def process_generated_arguments(run_name, en, container, alg, generated_arguments, selected_fitnes, selected_diversity, visited, outpath):
    visited_new = visited.copy()
    with concurrent.futures.ProcessPoolExecutor(max_workers=5) as executor:
        arg_futures={}        
        for args in generated_arguments:
            toupled = tuple(sorted(args.items()))
            if toupled in visited:
                future = Future().set_result(visited[toupled])

            future = executor.submit(
                evaluation_of_setup, 
                run_name=run_name,
                en=en, 
                alg=alg,
                ev_ng=20, 
                container=container,
                out_path=outpath,
                **args)
            arg_futures[future] = args
        max_fitness = selected_fitnes
        max_diversity = selected_diversity
        return_candidate = None
        for future in concurrent.futures.as_completed(arg_futures):
            args = arg_futures[future]
            toupled = tuple(sorted(args.items()))
            fitness, diversity  = future.result()
            visited_new[toupled] = (fitness, diversity)
            if fitness > max_fitness:
                print("We should have changed who's the best!")
                max_fitness = fitness
                return_candidate = args
                max_diversity = diversity
            elif fitness == max_fitness:
                if diversity > max_diversity:
                    print("We should have changed based on diversity!")
                    max_fitness = fitness
                    return_candidate = args
                    max_diversity = diversity
                
        return max_fitness, max_diversity, return_candidate, visited_new

            


In [17]:

def adaptive_lambda_grid_search(run_name, en,container, hops, starting_position, starting_fitness, dl, dm, dcr, dmr, outpath):
    visited = dict()
    visited[tuple(sorted(starting_position.items()))] = (starting_fitness, 0)
    selected = starting_position
    selected_fitness = starting_fitness
    selected_diversity = 0
    D_cr = [dcr, 0, -dcr]
    D_mr = [dmr, 0, -dmr]
    D_m = [dm, 0, -dm]
    D_l = [dl, 0, -dl]
    one_success = False
    for i in range(hops):

        generated_arguments = argument_combination_genration(
            selected=selected, 
            D_args={"cr":[D_cr, (0,1)], "l":[D_l, (0,None)]})
        # we have finised our 
        if generated_arguments == []: return selected, selected_fitness

        selection_fitness, selection_diversity, new_selected, new_visited = process_generated_arguments(
            run_name=run_name,
            en=en, 
            container=container, 
            alg="lambda",
            visited=visited, 
            generated_arguments=generated_arguments, 
            selected_fitnes=selected_fitness,
            selected_diversity=selected_diversity,
            outpath=outpath
        )
        if new_selected is not None:
            one_success = True
            if tuple(sorted(new_selected.items())) in visited:
                print("Cycle found! Terminating")
                return selected, new_visited
            selected = new_selected
            selected_diversity = selection_diversity
            selected_fitness = selection_fitness
        visited = new_visited
        generated_arguments = argument_combination_genration(
            selected=selected, 
            D_args={"mr":[D_mr, (0,1)], "m":[D_m, (0,None)]})
        # we have finised our 
        if generated_arguments == []: return selected, selected_fitness
        selection_fitness, selection_diversity, new_selected, new_visited = process_generated_arguments(
            run_name=run_name,
            en=en, 
            container=container, 
            alg="lambda", 
            generated_arguments=generated_arguments, 
            selected_fitnes=selected_fitness,
            selected_diversity=selected_diversity,
            visited=visited,
            outpath=outpath
        )
        if new_selected is None:
            visited = new_visited
        else:
            one_success = True
            if tuple(sorted(new_selected.items())) in visited:
                print("Cycle found! Terminating")
                return selected, new_visited
            visited = new_visited
            selected = new_selected
            selected_diversity = selection_diversity
            selected_fitness = selection_fitness
        
        if not one_success:
            return selected, visited
    return selected, visited
        
    
    



In [18]:

def adaptive_diff_grid_search(
        run_name,
        en,
        container, 
        hops, 
        starting_position, 
        starting_fitness, 
        dl, 
        dcr, 
        dmr, 
        outpath
    ):
    visited = dict()
    visited[tuple(sorted(starting_position.items()))] = (starting_fitness, 0)
    selected = starting_position
    selected_fitness = starting_fitness
    selected_diversity = 0
    D_cr = [dcr, 0, -dcr]
    D_mr = [dmr, 0, -dmr]
    D_l = [dl, 0, -dl]
    one_success = False
    for i in range(hops):

        generated_arguments = argument_combination_genration(
            selected=selected, 
            D_args={"l":[D_l, (0,None)]})
        # we have finised our 
        if generated_arguments == []: return selected, selected_fitness

        selection_fitness, selection_diversity, new_selected, new_visited = process_generated_arguments(
            run_name=run_name,
            en=en, 
            container=container, 
            alg="diff",
            visited=visited, 
            generated_arguments=generated_arguments, 
            selected_fitnes=selected_fitness,
            selected_diversity=selected_diversity,
            outpath=outpath
        )
        if new_selected is not None:
            one_success = True
            if tuple(sorted(new_selected.items())) in visited:
                print("Cycle found! Terminating")
                return selected, new_visited
            selected = new_selected
            selected_diversity = selection_diversity
            selected_fitness = selection_fitness
        visited = new_visited


        generated_arguments = argument_combination_genration(
            selected=selected, 
            D_args={"mr":[D_mr, (0,1)], "cr":[D_cr, (0,None)]})
        # we have finised our 
        if generated_arguments == []: return selected, selected_fitness
        selection_fitness, selection_diversity, new_selected, new_visited = process_generated_arguments(
            run_name=run_name,
            en=en, 
            container=container, 
            alg="diff",
            visited=visited, 
            generated_arguments=generated_arguments, 
            selected_fitnes=selected_fitness,
            selected_diversity=selected_diversity,
            outpath=outpath
        )
        if new_selected is None:
            visited = new_visited
        else:
            one_success = True
            if tuple(sorted(new_selected.items())) in visited:
                print("Cycle found! Terminating")
                return selected, new_visited
            visited = new_visited
            selected = new_selected
            selected_diversity = selection_diversity
            selected_fitness = selection_fitness
        
        if not one_success:
            return selected, visited
        
    return selected, visited

        
    
    



In [19]:
def adaptive_grid_search(en, alg, run_name, container, hops = 3, out_path="./Data/grid_search"):
    with open("relevant_studies.json", "r") as f:
        relevant_study_names = json.load(f)
    storage = f"sqlite:///Data/optuna/{en}/{container}/{alg}.db"
    study_name = relevant_study_names[container][alg][en]

    new_study = optuna.load_study(study_name=study_name,storage=storage)
    most_promising, mi = select_minimal_exaples([t.params for t in new_study.best_trials])
    selected_trial = new_study.best_trials[mi[0]]
    start =datetime.datetime.now()
    
    if alg=="lambda":
        if en == "lunarlander":
            dl = 15
            dm = 15
            dcr = 0.1
            dmr = 0.05
        else: #cartpole
            dl = 10
            dm = 10
            dcr = 0.1
            dmr = 0.05
        selected, visited = adaptive_lambda_grid_search(
            run_name=run_name,
            en=en,
            container=container, 
            hops=hops, 
            starting_position=rename(most_promising[0]),
            starting_fitness=selected_trial.value, 
            dl=dl, 
            dm=dm, 
            dcr=dcr, 
            dmr=dmr,
            outpath=out_path+f"/{en}"
        )
    elif alg =="diff":
        if en == "lunarlander":
            dl = 10
            dcr = 0.1
            dmr = 0.1
        else: #cartpole
            dl = 5
            dcr = 0.1
            dmr = 0.1
        selected, visited = adaptive_diff_grid_search(
            run_name=run_name,
            en=en,
            container=container, 
            hops=hops, 
            starting_position=rename(most_promising[0]),
            starting_fitness=selected_trial.value, 
            dl=dl,  
            dcr=dcr, 
            dmr=dmr,
            outpath=out_path+f"/{en}"
        )
    end =datetime.datetime.now()
    ts = datetime.datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    filename = f"{ts}_{container}_{en}_protocol.json"
    dirpath = os.path.join(os.path.realpath(out_path),en, container,alg)

    protocol = {"start":start.strftime("%Y-%m-%d_%H-%M-%S"), "end": end.strftime("%Y-%m-%d_%H-%M-%S"), "origin": most_promising, "final":selected, "visited":[dict(k) for k in visited]}
    json_path = os.path.join(dirpath, filename + ".json")
    with open(json_path, "w") as json_file:
        json.dump(protocol,json_file)


    



In [20]:
protocol = {}
#json_path = os.path.join(dirpath, filename + ".json")
json_path = "/home/schkliba/git/MastersThesis/Data/grid_search/first_try/lunarlander/novelty/lambda/2026-06-02_07-15-37_novelty_cross_method: uniform|l: 70|m: 40|mr: 0.06|cr: 0.7|mutation_sigma: 2.5|cross_uni: 0.2|ng: 20.json"
with open(json_path, "w") as json_file:
        json.dump(protocol,json_file)

In [21]:
adaptive_grid_search(
    en="cartpole", 
    alg="lambda", 
    container="novelty", run_name="novelty", hops = 3, out_path="./Data/grid_search/first_try")


Leaving out {'cross_method': 'uniform', 'l': 0, 'm': 20, 'mr': 0.08, 'cr': 0.4, 'mutation_sigma': 0.5, 'cross_uni': 0.30000000000000004}
Leaving out {'cross_method': 'uniform', 'l': 0, 'm': 20, 'mr': 0.08, 'cr': 0.3, 'mutation_sigma': 0.5, 'cross_uni': 0.30000000000000004}
Leaving out {'cross_method': 'uniform', 'l': 0, 'm': 20, 'mr': 0.08, 'cr': 0.19999999999999998, 'mutation_sigma': 0.5, 'cross_uni': 0.30000000000000004}
Launching lambdaon Enviroment cartpole with {'cross_method': 'uniform', 'l': 10, 'm': 20, 'mr': 0.08, 'cr': 0.4, 'mutation_sigma': 0.5, 'cross_uni': 0.30000000000000004, 'ng': 20}Launching lambdaon Enviroment cartpole with {'cross_method': 'uniform', 'l': 20, 'm': 20, 'mr': 0.08, 'cr': 0.3, 'mutation_sigma': 0.5, 'cross_uni': 0.30000000000000004, 'ng': 20}Launching lambdaon Enviroment cartpole with {'cross_method': 'uniform', 'l': 20, 'm': 20, 'mr': 0.08, 'cr': 0.19999999999999998, 'mutation_sigma': 0.5, 'cross_uni': 0.30000000000000004, 'ng': 20}Launching lambdaon E

2026-06-06 09:26:07.180699: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:998] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2026-06-06 09:26:07.185159: W tensorflow/core/common_runtime/gpu/gpu_device.cc:2251] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...
2026-06-06 09:26:07.189910: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:998] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/li

   	      	                    fitness                    	                                novelty                                 
   	      	-----------------------------------------------	------------------------------------------------------------------------
gen	nevals	avg    	gen	max    	min	nevals	std    	avg     	gen	max     	min     	nevals	std     
0  	10    	18.9667	0  	41.3333	8  	10    	12.1193	0.485193	0  	0.824636	0.346479	10    	0.207662
   	      	                    fitness                    	                            novelty                             
   	      	-----------------------------------------------	----------------------------------------------------------------
gen	nevals	avg    	gen	max    	min	nevals	std    	avg     	gen	max    	min     	nevals	std     
0  	10    	18.1333	0  	98.3333	9  	10    	26.7337	0.361959	0  	0.93749	0.224966	10    	0.274262
   	      	                    fitness                    	                            novelty        

/home/schkliba/.pyenv/versions/prometheus/lib/python3.10/site-packages/deap/creator.py:185: RuntimeWarning: A class named 'FitnessNovelty' has already been created and it will be overwritten. Consider deleting previous creation of that class or rename it.
  warnings.warn("A class named '{0}' has already been created and it "
/home/schkliba/.pyenv/versions/prometheus/lib/python3.10/site-packages/deap/creator.py:185: RuntimeWarning: A class named 'FitnessNovelty' has already been created and it will be overwritten. Consider deleting previous creation of that class or rename it.
  warnings.warn("A class named '{0}' has already been created and it "
/home/schkliba/.pyenv/versions/prometheus/lib/python3.10/site-packages/deap/creator.py:185: RuntimeWarning: A class named 'FitnessNovelty' has already been created and it will be overwritten. Consider deleting previous creation of that class or rename it.
  warnings.warn("A class named '{0}' has already been created and it "
/home/schkliba/.pye

16 	11    	9.66667	16 	9.66667	9.66667	11    	1.77636e-15	0.910315	16 	0.910315	0.910315	11    	1.11022e-16


2026-06-06 09:27:48.708235: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:998] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2026-06-06 09:27:48.713168: W tensorflow/core/common_runtime/gpu/gpu_device.cc:2251] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...
2026-06-06 09:27:48.733180: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:998] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/li

13 	7     	10.1333	13 	10.3333	9.66667	7     	0.305505	0.908451	13 	0.90898 	0.907657	7     	0.00064788
12 	10    	10     	12 	10     	10     	10    	0       	0.885626	12 	0.885626	0.885626	10    	0        
17 	13    	9.76667	17 	10.6667	9.66667	13    	0.3        	1.01486 	17 	1.05695 	0.636104	13    	0.126253   
9  	9     	9.66667	9  	9.66667	9.66667	9     	1.77636e-15	1.07306 	9  	1.07306 	1.07306 	9     	2.22045e-16
17 	7     	9.66667	17 	9.66667	9.66667	7     	1.77636e-15	0.910315	17 	0.910315	0.910315	7     	1.11022e-16
11 	8     	10.1667	11 	10.3333	9.66667	8     	0.288675	0.94806 	11 	0.967467	0.935122	8     	0.0158456
   	      	                    fitness                    	                            novelty                             
   	      	-----------------------------------------------	----------------------------------------------------------------
gen	nevals	avg    	gen	max    	min	nevals	std    	avg     	gen	max    	min     	nevals	std     
0  	10    	18.1333	0  

BrokenProcessPool: A process in the process pool was terminated abruptly while the future was running or pending.

In [ ]:
from cartpole import CartpoleEvaluator
CartpoleEvaluator.hidden_dim

4